In [ ]:
import google.auth
credentials, project = google.auth.default()
print(credentials)

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="my-dissertation-project-502620")

# query = """
# SELECT * FROM `physionet-data.mimiciv_3_1_hosp.patients` LIMIT 5
# """
# The following query select the potential interesting cases that the agents will find it useful to debate about.
# The criteria used to select the patients along with their hadm is as follow:
#• ICU admission ≥ 48 hours 
#• Long hospital stay ≥ 5 days 
#• Multiple diagnoses ≥ 4 ICD codes 
#• Rich laboratory history ≥ 50 results 
#• Medication history ≥ 10 prescriptions 
#• Multiple clinical notes ≥ 5 notes
query = """
WITH diagnosis_counts AS (
  SELECT
    hadm_id,
    COUNT(*) AS diagnosis_count
  FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd`
  GROUP BY hadm_id
),

lab_counts AS (
  SELECT
    hadm_id,
    COUNT(*) AS lab_count
  FROM `physionet-data.mimiciv_3_1_hosp.labevents`
  GROUP BY hadm_id
),

medication_counts AS (
  SELECT
    hadm_id,
    COUNT(*) AS medication_count
  FROM `physionet-data.mimiciv_3_1_hosp.prescriptions`
  GROUP BY hadm_id
),

note_counts AS (
  SELECT
    hadm_id,
    COUNT(*) AS note_count
  FROM (
    SELECT hadm_id
    FROM `physionet-data.mimiciv_note.discharge`

    UNION ALL

    SELECT hadm_id
    FROM `physionet-data.mimiciv_note.radiology`
  )
  GROUP BY hadm_id
),

icu_hours AS (
  SELECT
    hadm_id,
    SUM(TIMESTAMP_DIFF(outtime, intime, HOUR)) AS icu_hours
  FROM `physionet-data.mimiciv_3_1_icu.icustays`
  WHERE outtime IS NOT NULL
  GROUP BY hadm_id
)

SELECT

    a.subject_id,
    a.hadm_id,

    DATE_DIFF(a.dischtime, a.admittime, DAY) AS hospital_days,

    IFNULL(i.icu_hours,0) AS icu_hours,

    d.diagnosis_count,

    l.lab_count,

    m.medication_count,

    n.note_count

FROM `physionet-data.mimiciv_3_1_hosp.admissions` a

JOIN diagnosis_counts d
ON a.hadm_id=d.hadm_id

JOIN lab_counts l
ON a.hadm_id=l.hadm_id

JOIN medication_counts m
ON a.hadm_id=m.hadm_id

JOIN note_counts n
ON a.hadm_id=n.hadm_id

LEFT JOIN icu_hours i
ON a.hadm_id=i.hadm_id

WHERE

DATE_DIFF(a.dischtime, a.admittime, DAY) >= 5

AND IFNULL(i.icu_hours,0) >= 48

AND d.diagnosis_count >= 4

AND l.lab_count >= 50

AND m.medication_count >= 10

AND n.note_count >= 5

ORDER BY

hospital_days DESC,
icu_hours DESC,
diagnosis_count DESC,
lab_count DESC;
"""
df = client.query(query).to_dataframe()
table_id = "my-dissertation-project-502620.Query_Results.candidate_cases"
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(df, table_id, job_config=job_config).result()
print(f"Loaded {len(df)} candidate cases to {table_id}")
#print(df)

In [ ]:
import pandas as pd
from google.cloud import bigquery

# ICD10_to_9.csv columns: icd10_code, icd9_code, rank (lower rank = closer match)
# gem = pd.read_csv(
#     "ICD10_to_9.csv",   # or full path, e.g. /home/jupyter/ICD10_to_9.csv
#     header=None, names=["icd10_code", "icd9_code", "rank"]
# )

# # keep the single best ICD10 match per ICD9 code
# gem_best = (gem.sort_values("rank")
#                .drop_duplicates(subset="icd9_code", keep="first")
#                [["icd9_code", "icd10_code"]])

# client = bigquery.Client()
# table_id = "my-dissertation-project-502620.Query_Results.icd9_to_icd10_gem"
# client.load_table_from_dataframe(gem_best, table_id).result()
# print(f"Loaded {len(gem_best)} ICD9->ICD10 mappings")



# gem = pd.read_csv(
#     "ICD10_to_9.csv",
#     header=None, names=["icd10_code", "icd9_code", "rank"]
# )
# gem["icd9_code"] = gem["icd9_code"].str.replace(".", "", regex=False)
# gem["icd10_code"] = gem["icd10_code"].str.replace(".", "", regex=False)

# gem_best = (gem.sort_values("rank")
#                .drop_duplicates(subset="icd9_code", keep="first")
#                [["icd9_code", "icd10_code"]])

# client = bigquery.Client()
# table_id = "my-dissertation-project-502620.Query_Results.icd9_to_icd10_gem"
# job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
# client.load_table_from_dataframe(gem_best, table_id, job_config=job_config).result()
# print(f"Loaded {len(gem_best)} ICD9->ICD10 mappings")



!pip install icd-mappings --break-system-packages -q

from icdmappings import Mapper
mapper = Mapper()

# Get distinct ICD-9 codes present in your cohort
query = r"""
SELECT DISTINCT di.icd_code AS icd9_code
FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` di
JOIN `my-dissertation-project-502620.Query_Results.candidate_cases` cc
  ON di.subject_id = cc.subject_id AND di.hadm_id = cc.hadm_id
WHERE di.icd_version = 9
"""
icd9_codes = client.query(query).to_dataframe()["icd9_code"].tolist()

icd10_mapped = mapper.map(icd9_codes, source="icd9", target="icd10")

gem_best = pd.DataFrame({
    "icd9_code": icd9_codes,
    "icd10_code": icd10_mapped
}).dropna(subset=["icd10_code"])

table_id = "my-dissertation-project-502620.Query_Results.icd9_to_icd10_gem"
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(gem_best, table_id, job_config=job_config).result()
print(f"Loaded {len(gem_best)} ICD9->ICD10 mappings")


In [ ]:
query = r"""
SELECT
  di.icd_version,
  COUNT(DISTINCT CONCAT(di.subject_id, '-', di.hadm_id)) AS n_admissions,
  COUNTIF(di.icd_version = 9 AND gem.icd10_code IS NULL) AS unmapped_icd9_rows
FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` di
JOIN `my-dissertation-project-502620.Query_Results.candidate_cases` cc
  ON di.subject_id = cc.subject_id AND di.hadm_id = cc.hadm_id
LEFT JOIN `my-dissertation-project-502620.Query_Results.icd9_to_icd10_gem` gem
  ON di.icd_version = 9 AND di.icd_code = gem.icd9_code
GROUP BY di.icd_version
"""
print(client.query(query).to_dataframe())

In [ ]:
# The following query narrow the previousely selected records to records related to:
# Cardiology, Nephrology, Infectious Diseases, Pulmonology, Moderator (or Critical Care if you prefer a clinical moderator)
query = r"""
WITH mapped_icd AS (
  SELECT
    di.subject_id,
    di.hadm_id,
    CASE
      WHEN di.icd_version = 10 THEN di.icd_code
      ELSE gem.icd10_code
    END AS icd10_code
  FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` di
  LEFT JOIN `my-dissertation-project-502620.Query_Results.icd9_to_icd10_gem` gem
    ON di.icd_version = 9 AND di.icd_code = gem.icd9_code
  JOIN `my-dissertation-project-502620.Query_Results.candidate_cases` cc
    ON di.subject_id = cc.subject_id AND di.hadm_id = cc.hadm_id
),

specialty_matches AS (
  SELECT subject_id, hadm_id, 'Cardiology' AS specialty
  FROM mapped_icd
  WHERE icd10_code IS NOT NULL
    AND (STARTS_WITH(icd10_code,'I0') OR STARTS_WITH(icd10_code,'I1')
      OR STARTS_WITH(icd10_code,'I2') OR STARTS_WITH(icd10_code,'I3')
      OR STARTS_WITH(icd10_code,'I4') OR STARTS_WITH(icd10_code,'I5'))

  UNION ALL

  SELECT subject_id, hadm_id, 'Nephrology' AS specialty
  FROM mapped_icd
  WHERE icd10_code IS NOT NULL
    AND (STARTS_WITH(icd10_code,'N0') OR STARTS_WITH(icd10_code,'N1')
      OR STARTS_WITH(icd10_code,'N2'))

  UNION ALL

  SELECT subject_id, hadm_id, 'Infectious Diseases' AS specialty
  FROM mapped_icd
  WHERE icd10_code IS NOT NULL
    AND (STARTS_WITH(icd10_code,'A') OR STARTS_WITH(icd10_code,'B'))

  UNION ALL

  SELECT subject_id, hadm_id, 'Pulmonology' AS specialty
  FROM mapped_icd
  WHERE icd10_code IS NOT NULL
    AND STARTS_WITH(icd10_code,'J')
)

SELECT
    subject_id,
    hadm_id,
    ARRAY_AGG(DISTINCT specialty ORDER BY specialty) AS specialties,
    COUNT(DISTINCT specialty) AS num_specialties
FROM specialty_matches
GROUP BY subject_id, hadm_id
HAVING COUNT(DISTINCT specialty) >= 2
ORDER BY num_specialties DESC, subject_id, hadm_id;
"""
df1 = client.query(query).to_dataframe()
print(df1.shape)
print(df1['num_specialties'].value_counts().sort_index())




# Saving the narrowed candidate cases based on specialty into a new table (narrowed_candidate_cases) in the Query_Results database
df1 = client.query(query).to_dataframe()
table_id = "my-dissertation-project-502620.Query_Results.narrowed_candidate_cases"
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(df1, table_id, job_config=job_config).result()
print(f"Loaded {len(df1)} candidate cases to {table_id}")
#print(df)

In [ ]:
query = "SELECT COUNT(*) AS n FROM `my-dissertation-project-502620.Query_Results.candidate_cases`"
total = client.query(query).to_dataframe()["n"][0]
print(f"{20447}/{total} = {20447/total:.1%}")

In [ ]:
query_disease_frequency = r"""
WITH four_specialty_cases AS (
    SELECT DISTINCT
        subject_id,
        hadm_id
    FROM `my-dissertation-project-502620.Query_Results.narrowed_candidate_cases`
    WHERE num_specialties = 4
),

case_diagnoses AS (
    SELECT DISTINCT
        di.subject_id,
        di.hadm_id,
        di.icd_code,
        di.icd_version
    FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` di
    INNER JOIN four_specialty_cases fc
        ON di.subject_id = fc.subject_id
       AND di.hadm_id = fc.hadm_id
)

SELECT
    cd.icd_code,
    cd.icd_version,
    d.long_title,
    COUNT(DISTINCT cd.hadm_id) AS admission_count,
    ROUND(
        100 * COUNT(DISTINCT cd.hadm_id) /
        (SELECT COUNT(*) FROM four_specialty_cases),
        2
    ) AS percentage_of_cases
FROM case_diagnoses cd
LEFT JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses` d
    ON cd.icd_code = d.icd_code
   AND cd.icd_version = d.icd_version
GROUP BY
    cd.icd_code,
    cd.icd_version,
    d.long_title
ORDER BY
    admission_count DESC;
"""

df_disease_frequency = client.query(query_disease_frequency).to_dataframe()

print(df_disease_frequency.shape)
display(df_disease_frequency.head(50))

In [ ]:
query_normalized_frequency = r"""
WITH four_specialty_cases AS (
    SELECT DISTINCT
        subject_id,
        hadm_id
    FROM `my-dissertation-project-502620.Query_Results.narrowed_candidate_cases`
    WHERE num_specialties = 4
),

normalized_diagnoses AS (
    SELECT DISTINCT
        di.subject_id,
        di.hadm_id,

        CASE
            WHEN di.icd_version = 10 THEN di.icd_code
            WHEN di.icd_version = 9 THEN gem.icd10_code
        END AS icd10_code

    FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` di

    INNER JOIN four_specialty_cases fc
        ON di.subject_id = fc.subject_id
       AND di.hadm_id = fc.hadm_id

    LEFT JOIN `my-dissertation-project-502620.Query_Results.icd9_to_icd10_gem` gem
        ON di.icd_version = 9
       AND di.icd_code = gem.icd9_code
),

valid_diagnoses AS (
    SELECT DISTINCT
        subject_id,
        hadm_id,
        icd10_code
    FROM normalized_diagnoses
    WHERE icd10_code IS NOT NULL
)

SELECT
    vd.icd10_code,
    d.long_title,

    COUNT(DISTINCT vd.hadm_id) AS admission_count,

    ROUND(
        100 * COUNT(DISTINCT vd.hadm_id)
        / (SELECT COUNT(*) FROM four_specialty_cases),
        2
    ) AS percentage_of_cases

FROM valid_diagnoses vd

LEFT JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses` d
    ON vd.icd10_code = d.icd_code
   AND d.icd_version = 10

GROUP BY
    vd.icd10_code,
    d.long_title

ORDER BY
    admission_count DESC
"""

df_normalized_frequency = client.query(
    query_normalized_frequency
).to_dataframe()

print(df_normalized_frequency.shape)

#display(df_normalized_frequency.head(100))
print(
    df_normalized_frequency.head(100).to_string(index=False)
)

In [ ]:
%pip install openpyxl

In [ ]:
# loading the CCSR Excel file
import pandas as pd

ccsr_file = "DXCCSR-Reference-File-v2026-1.xlsx"

df_ccsr = pd.read_excel(
    ccsr_file,
    sheet_name="DX_to_CCSR_Mapping",
    header=1
)

print(df_ccsr.shape)
print(df_ccsr.columns.tolist())
df_ccsr.head()

In [ ]:
# Retrieving the normalized diagnoses for the 5,125 four-specialty admissions:
query_normalized_diagnoses = r"""
WITH four_specialty_cases AS (
    SELECT DISTINCT
        subject_id,
        hadm_id
    FROM `my-dissertation-project-502620.Query_Results.narrowed_candidate_cases`
    WHERE num_specialties = 4
),

normalized_diagnoses AS (
    SELECT DISTINCT
        di.subject_id,
        di.hadm_id,
        CASE
            WHEN di.icd_version = 10 THEN di.icd_code
            WHEN di.icd_version = 9 THEN gem.icd10_code
        END AS icd10_code
    FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` di

    INNER JOIN four_specialty_cases fc
        ON di.subject_id = fc.subject_id
       AND di.hadm_id = fc.hadm_id

    LEFT JOIN `my-dissertation-project-502620.Query_Results.icd9_to_icd10_gem` gem
        ON di.icd_version = 9
       AND di.icd_code = gem.icd9_code
)

SELECT DISTINCT
    subject_id,
    hadm_id,
    icd10_code
FROM normalized_diagnoses
WHERE icd10_code IS NOT NULL
"""

df_normalized_diagnoses = client.query(
    query_normalized_diagnoses
).to_dataframe()

print(df_normalized_diagnoses.shape)
df_normalized_diagnoses.head()

In [ ]:
# Clean the code formatting and join with CCSR
# Standardise ICD-10 code formatting
df_normalized_diagnoses["icd10_code"] = (
    df_normalized_diagnoses["icd10_code"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.strip()
    .str.upper()
)

df_ccsr["ICD-10-CM Code"] = (
    df_ccsr["ICD-10-CM Code"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.strip()
    .str.upper()
)

# Map each ICD-10 code to ALL applicable CCSR categories
df_diagnoses_ccsr = df_normalized_diagnoses.merge(
    df_ccsr[
        [
            "ICD-10-CM Code",
            "CCSR Category",
            "CCSR Category Description",
            "Inpatient Default CCSR (Y/N/X)"
        ]
    ],
    left_on="icd10_code",
    right_on="ICD-10-CM Code",
    how="left"
)

# Remove duplicate admission + ICD + CCSR combinations
df_diagnoses_ccsr = df_diagnoses_ccsr.drop_duplicates(
    subset=["hadm_id", "icd10_code", "CCSR Category"]
)

print(df_diagnoses_ccsr.shape)
df_diagnoses_ccsr.head(20)

In [ ]:
print(df_diagnoses_ccsr.shape)
print(df_diagnoses_ccsr["CCSR Category"].isna().sum())
df_diagnoses_ccsr.head(20)

In [ ]:
# Investigating the unmapped ICD-10 codes
unmapped = (
    df_diagnoses_ccsr[
        df_diagnoses_ccsr["CCSR Category"].isna()
    ][["icd10_code"]]
    .drop_duplicates()
)

print("Unique unmapped ICD-10 codes:", len(unmapped))
print(unmapped.to_string(index=False))

In [ ]:
df_diagnoses_ccsr[
    df_diagnoses_ccsr["CCSR Category"].isna()
]["icd10_code"].value_counts()

In [ ]:
# Keep only diagnoses successfully mapped to a CCSR category
df_diagnoses_ccsr_mapped = df_diagnoses_ccsr[
    df_diagnoses_ccsr["CCSR Category"].notna()
].copy()

print(df_diagnoses_ccsr_mapped.shape)
print(df_diagnoses_ccsr_mapped["CCSR Category"].isna().sum())

In [ ]:
df_admission_ccsr = (
    df_diagnoses_ccsr_mapped[
        ["hadm_id", "CCSR Category", "CCSR Category Description"]
    ]
    .drop_duplicates(
        subset=["hadm_id", "CCSR Category"]
    )
)

print(df_admission_ccsr.shape)
df_admission_ccsr.head()

In [ ]:
total_admissions = 5125

df_ccsr_frequency = (
    df_admission_ccsr
    .groupby(
        ["CCSR Category", "CCSR Category Description"]
    )
    .size()
    .reset_index(name="admission_count")
)

df_ccsr_frequency["percentage_of_cases"] = (
    100 * df_ccsr_frequency["admission_count"] / total_admissions
).round(2)

df_ccsr_frequency = (
    df_ccsr_frequency
    .sort_values("admission_count", ascending=False)
    .reset_index(drop=True)
)

print(df_ccsr_frequency.shape)
df_ccsr_frequency.head(30)

In [ ]:
specialty_prefixes = {
    "Cardiology": "CIR",
    "Pulmonology": "RSP",
    "Nephrology": "GEN",
    "Infectious Diseases": "INF"
}

specialty_tables = {}

for specialty, prefix in specialty_prefixes.items():

    df_temp = df_ccsr_frequency[
        df_ccsr_frequency["CCSR Category"].str.startswith(prefix)
    ].copy()

    df_temp["specialty"] = specialty

    specialty_tables[specialty] = df_temp

    print(f"\n{specialty}: {len(df_temp)} CCSR categories")
    print(
        df_temp[
            [
                "CCSR Category",
                "CCSR Category Description",
                "admission_count",
                "percentage_of_cases"
            ]
        ].head(15).to_string(index=False)
    )

In [ ]:
# Start with all CCSR families and their frequencies
df_specialty_eligibility = df_ccsr_frequency.copy()

# Create four eligibility columns
df_specialty_eligibility["Cardiology"] = False
df_specialty_eligibility["Pulmonology"] = False
df_specialty_eligibility["Nephrology"] = False
df_specialty_eligibility["Infectious_Diseases"] = False


# -------------------------
# 1. Primary CCSR rules
# -------------------------

# Circulatory system → Cardiology candidates
df_specialty_eligibility.loc[
    df_specialty_eligibility["CCSR Category"].str.startswith("CIR"),
    "Cardiology"
] = True

# Respiratory system → Pulmonology candidates
df_specialty_eligibility.loc[
    df_specialty_eligibility["CCSR Category"].str.startswith("RSP"),
    "Pulmonology"
] = True

# Infectious disease system → ID candidates
df_specialty_eligibility.loc[
    df_specialty_eligibility["CCSR Category"].str.startswith("INF"),
    "Infectious_Diseases"
] = True


# -------------------------
# 2. Nephrology candidates
# -------------------------

# For now, mark GEN categories as candidates.
# We will subsequently remove non-renal GEN categories
# (e.g. prostate/female genital conditions).
df_specialty_eligibility.loc[
    df_specialty_eligibility["CCSR Category"].str.startswith("GEN"),
    "Nephrology"
] = True


# -------------------------
# Show only candidate families
# -------------------------

df_specialty_candidates = df_specialty_eligibility[
    df_specialty_eligibility[
        ["Cardiology", "Pulmonology",
         "Nephrology", "Infectious_Diseases"]
    ].any(axis=1)
].copy()

print(df_specialty_candidates.shape)

print(
    df_specialty_candidates[
        [
            "CCSR Category",
            "CCSR Category Description",
            "admission_count",
            "percentage_of_cases",
            "Cardiology",
            "Pulmonology",
            "Nephrology",
            "Infectious_Diseases"
        ]
    ].to_string(index=False)
)

In [ ]:
# Categories from the four main CCSR systems
main_prefixes = ("CIR", "RSP", "GEN", "INF")

df_scope_review = df_ccsr_frequency[
    df_ccsr_frequency["CCSR Category"].str.startswith(main_prefixes)
].copy()

# Add potentially important categories outside those systems
keywords = (
    "fluid|electrolyte|acid|potassium|sodium|"
    "infection|infectious|sepsis|"
    "shock"
)

df_extra_scope = df_ccsr_frequency[
    df_ccsr_frequency["CCSR Category Description"].str.contains(
        keywords,
        case=False,
        na=False,
        regex=True
    )
].copy()

# Combine without duplicates
df_scope_review = (
    pd.concat([df_scope_review, df_extra_scope])
    .drop_duplicates(subset=["CCSR Category"])
    .sort_values(["CCSR Category"])
    .reset_index(drop=True)
)

print("Categories to review:", len(df_scope_review))

print(
    df_scope_review[
        [
            "CCSR Category",
            "CCSR Category Description"
        ]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# SPECIALTY ELIGIBILITY MAPPING
# CCSR defines the disease families.
# This mapping defines which specialist(s) cover each family.
# ============================================================

specialty_ccsr = {

    # -------------------------
    # CARDIOLOGY
    # -------------------------
    "Cardiology": [
        "CIR001",  # Chronic rheumatic heart disease
        "CIR002",  # Acute rheumatic heart disease
        "CIR003",  # Nonrheumatic/unspecified valve disorders
        "CIR004",  # Endocarditis/endocardial disease
        "CIR005",  # Myocarditis and cardiomyopathy
        "CIR006",  # Pericarditis/pericardial disease
        "CIR007",  # Essential hypertension
        "CIR008",  # Complicated/secondary hypertension
        "CIR009",  # Acute myocardial infarction
        "CIR010",  # Complications of acute MI
        "CIR011",  # Coronary atherosclerosis/other heart disease
        "CIR013",  # Acute pulmonary embolism
        "CIR014",  # Pulmonary heart disease
        "CIR015",  # Other/ill-defined heart disease
        "CIR016",  # Conduction disorders
        "CIR017",  # Cardiac dysrhythmias
        "CIR018",  # Cardiac arrest/ventricular fibrillation
        "CIR019",  # Heart failure
        "CIR026",  # Peripheral/visceral vascular disease
        "CIR027",  # Arterial dissections
        "CIR029",  # Aortic/peripheral/visceral aneurysms
        "CIR030",  # Arterial embolism/thrombosis
        "CIR033",  # Acute phlebitis/thromboembolism
        "CIR034",  # Chronic phlebitis/thromboembolism
        "CIR036",  # Postthrombotic syndrome/venous insufficiency
    ],

    # -------------------------
    # PULMONOLOGY
    # -------------------------
    "Pulmonology": [
        "RSP002",  # Pneumonia
        "RSP003",  # Influenza
        "RSP005",  # Acute bronchitis
        "RSP008",  # COPD and bronchiectasis
        "RSP009",  # Asthma
        "RSP010",  # Aspiration pneumonitis
        "RSP011",  # Pleurisy/pleural effusion/pulmonary collapse
        "RSP012",  # Respiratory failure
        "RSP013",  # Lung disease due to external agents
        "RSP014",  # Pneumothorax
        "RSP015",  # Mediastinal disorders
        "RSP016",  # Other lower respiratory disease
    ],

    # -------------------------
    # NEPHROLOGY
    # -------------------------
    "Nephrology": [
        "GEN001",  # Nephritis/nephrosis/renal sclerosis
        "GEN002",  # Acute/unspecified renal failure
        "GEN003",  # Chronic kidney disease
        "GEN005",  # Urinary tract calculus
        "GEN006",  # Other kidney/ureter disease
        "GEN009",  # Hematuria
        "GEN010",  # Proteinuria
        "END011",  # Fluid and electrolyte disorders
    ],

    # -------------------------
    # INFECTIOUS DISEASES
    # -------------------------
    "Infectious Diseases": [
        "INF001",  # Tuberculosis
        "INF002",  # Septicemia
        "INF003",  # Bacterial infections
        "INF004",  # Fungal infections
        "INF005",  # Foodborne intoxications
        "INF006",  # HIV
        "INF007",  # Hepatitis
        "INF008",  # Viral infection
        "INF009",  # Parasitic/other infections
        "INF010",  # Sexually transmitted infections
        "INF011",  # Sequela of infectious diseases

        # Organ-specific infectious families
        "RSP002",  # Pneumonia
        "RSP003",  # Influenza
        "RSP005",  # Acute bronchitis
        "RSP006",  # Other upper respiratory infections
        "GEN004",  # Urinary tract infections
        "DIG001",  # Intestinal infection
        "NVS003",  # CNS infection
        "SKN001",  # Skin/subcutaneous tissue infections
    ]
}

In [ ]:
specialty_results = {}

for specialty, eligible_codes in specialty_ccsr.items():

    df_temp = df_ccsr_frequency[
        df_ccsr_frequency["CCSR Category"].isin(eligible_codes)
    ].copy()

    # Frequency determines ranking
    df_temp = (
        df_temp
        .sort_values(
            "admission_count",
            ascending=False
        )
        .reset_index(drop=True)
    )

    df_temp["rank"] = range(1, len(df_temp) + 1)

    specialty_results[specialty] = df_temp

    print(f"\n{'='*70}")
    print(specialty.upper())
    print(f"{'='*70}")

    print(
        df_temp[
            [
                "rank",
                "CCSR Category",
                "CCSR Category Description",
                "admission_count",
                "percentage_of_cases"
            ]
        ].head(10).to_string(index=False)
    )

In [ ]:
# We will calculate how many of the 5,125 admissions satisfy 4/4 top-5 specialty coverage
# Step 1 — Define the frozen top-5 families
top5_specialty_ccsr = {
    "Cardiology": [
        "CIR017",  # Cardiac dysrhythmias
        "CIR019",  # Heart failure
        "CIR008",  # Complicated/secondary hypertension
        "CIR011",  # Coronary atherosclerosis/other heart disease
        "CIR007",  # Essential hypertension
    ],

    "Pulmonology": [
        "RSP012",  # Respiratory failure
        "RSP002",  # Pneumonia
        "RSP008",  # COPD/bronchiectasis
        "RSP011",  # Pleural effusion/pulmonary collapse
        "RSP010",  # Aspiration pneumonitis
    ],

    "Nephrology": [
        "GEN002",  # Acute/unspecified renal failure
        "END011",  # Fluid/electrolyte disorders
        "GEN003",  # Chronic kidney disease
        "GEN006",  # Other kidney/ureter diseases
        "GEN001",  # Nephritis/nephrosis/renal sclerosis
    ],

    "Infectious Diseases": [
        "INF003",  # Bacterial infections
        "INF002",  # Septicemia
        "RSP002",  # Pneumonia
        "GEN004",  # UTI
        "INF004",  # Fungal infections
    ]
}

In [ ]:
# Step 2 — Determine specialty coverage for every admission
coverage_rows = []

for specialty, codes in top5_specialty_ccsr.items():

    admissions = df_admission_ccsr[
        df_admission_ccsr["CCSR Category"].isin(codes)
    ]["hadm_id"].unique()

    coverage_rows.extend(
        {"hadm_id": hadm_id, "specialty": specialty}
        for hadm_id in admissions
    )

df_top5_coverage = pd.DataFrame(coverage_rows)

df_specialty_count = (
    df_top5_coverage
    .drop_duplicates(["hadm_id", "specialty"])
    .groupby("hadm_id")
    .size()
    .reset_index(name="num_top5_specialties")
)

In [ ]:
#Step 3 — See how many admissions have 1/4, 2/4, 3/4 and 4/4
coverage_summary = (
    df_specialty_count["num_top5_specialties"]
    .value_counts()
    .sort_index()
)

print(coverage_summary)

In [ ]:
# Checking how many admissions with all 4 top-5 specialties 
four_specialty_cases = df_specialty_count[
    df_specialty_count["num_top5_specialties"] == 4
].copy()

print("Admissions with all 4 top-5 specialties:",
      len(four_specialty_cases))

In [ ]:
# Now we'll measure how common each of the 20 specialist-family positions is within the 4,591 admissions that satisfy 4/4 coverage.
# Step 1 — Get the 4/4 admission IDs
four_specialty_ids = set(
    four_specialty_cases["hadm_id"]
)

# Step 2 — Calculate top-5 family frequencies within those 4,591 cases
top5_distribution = []

for specialty, codes in top5_specialty_ccsr.items():

    for code in codes:

        # Admissions containing this CCSR family
        admissions_with_family = df_admission_ccsr[
            (df_admission_ccsr["hadm_id"].isin(four_specialty_ids)) &
            (df_admission_ccsr["CCSR Category"] == code)
        ]["hadm_id"].nunique()

        description = (
            df_admission_ccsr.loc[
                df_admission_ccsr["CCSR Category"] == code,
                "CCSR Category Description"
            ]
            .iloc[0]
        )

        top5_distribution.append({
            "specialty": specialty,
            "CCSR Category": code,
            "CCSR Category Description": description,
            "admission_count": admissions_with_family,
            "percentage_of_4591": round(
                100 * admissions_with_family / len(four_specialty_ids),
                2
            )
        })

df_top5_distribution = pd.DataFrame(top5_distribution)

print(
    df_top5_distribution.to_string(index=False)
)

In [ ]:
import matplotlib.pyplot as plt

# Create a copy for plotting
df_plot = df_top5_distribution.copy()

# Shorter labels for readability
df_plot["label"] = (
    df_plot["CCSR Category"] + " - " +
    df_plot["CCSR Category Description"]
)

# Plot each specialty separately
for specialty in df_plot["specialty"].unique():

    df_s = df_plot[
        df_plot["specialty"] == specialty
    ].copy()

    # Reverse so rank 1 appears at the top
    df_s = df_s.iloc[::-1]

    plt.figure(figsize=(10, 2.5))
    plt.barh(
        df_s["label"],
        df_s["percentage_of_4591"],
        height=0.35
    )

    plt.xlabel("Percentage of eligible admissions (%)")
    plt.ylabel("CCSR disease family")
    plt.title(
        f"Prevalence of Top-5 CCSR Disease Families: {specialty}"
    )

    plt.xlim(0, 100)

    # Add percentage labels
    for i, value in enumerate(df_s["percentage_of_4591"]):
        plt.text(
            value + 1,
            i,
            f"{value:.1f}%",
            va="center"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
# The implementation of the coverage-aware selection of 100 cases.
#Step 1 — Create the 19 unique target families
target_codes = sorted(
    set(
        code
        for codes in top5_specialty_ccsr.values()
        for code in codes
    )
)

print("Number of unique target families:", len(target_codes))
print(target_codes)


In [ ]:
# Step 2 — Create the eligible case × disease matrix
eligible_matrix = (
    df_admission_ccsr[
        (df_admission_ccsr["hadm_id"].isin(four_specialty_ids)) &
        (df_admission_ccsr["CCSR Category"].isin(target_codes))
    ]
    .assign(present=1)
    .pivot_table(
        index="hadm_id",
        columns="CCSR Category",
        values="present",
        aggfunc="max",
        fill_value=0
    )
)

# Ensure all target columns exist
eligible_matrix = eligible_matrix.reindex(
    columns=target_codes,
    fill_value=0
)

print(eligible_matrix.shape)

In [ ]:
#Step 3 — Select cases to ensure ≥15 per family
import numpy as np

RANDOM_SEED = 42
SAMPLE_SIZE = 100
MIN_PER_FAMILY = 15

rng = np.random.default_rng(RANDOM_SEED)

remaining = eligible_matrix.copy()
selected_ids = []

coverage = {code: 0 for code in target_codes}

while min(coverage.values()) < MIN_PER_FAMILY:

    # Families that still need more cases
    needed = [
        code for code in target_codes
        if coverage[code] < MIN_PER_FAMILY
    ]

    # Among the needed families, find the one with the
    # fewest remaining available admissions
    remaining_availability = {
        code: int(remaining[code].sum())
        for code in needed
    }

    target_family = min(
        needed,
        key=lambda code: remaining_availability[code]
    )

    # Only consider admissions containing this rare/limiting family
    candidates = remaining[
        remaining[target_family] == 1
    ]

    if candidates.empty:
        raise ValueError(
            f"Cannot achieve minimum coverage for {target_family}"
        )

    # Prefer a candidate that simultaneously covers
    # the largest number of other still-needed families
    scores = candidates[needed].sum(axis=1)

    max_score = scores.max()

    best_candidates = scores[
        scores == max_score
    ].index.to_numpy()

    # Random tie-break, but reproducible
    chosen = rng.choice(best_candidates)

    selected_ids.append(chosen)

    # Update coverage
    for code in target_codes:
        if remaining.loc[chosen, code] == 1:
            coverage[code] += 1

    remaining = remaining.drop(index=chosen)


print("Cases needed to achieve minimum coverage:", len(selected_ids))

print("\nCoverage after Phase 1:")
for code in target_codes:
    print(code, coverage[code])

In [ ]:
#Step 4 — Randomly fill the remainder to 100
n_remaining = SAMPLE_SIZE - len(selected_ids)

additional_ids = rng.choice(
    remaining.index.to_numpy(),
    size=n_remaining,
    replace=False
)

selected_ids.extend(additional_ids.tolist())

print("\nFinal number of selected cases:",
      len(selected_ids))

In [ ]:
# Step 5 — Check the final disease distribution
final_matrix = eligible_matrix.loc[selected_ids]

final_distribution = (
    final_matrix
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

final_distribution.columns = [
    "CCSR Category",
    "cases_in_final_100"
]

final_distribution["percentage"] = (
    final_distribution["cases_in_final_100"]
).round(2)   # denominator is 100

print(final_distribution.to_string(index=False))

In [ ]:
assert set(selected_ids).issubset(four_specialty_ids)
assert len(selected_ids) == 100
assert len(set(selected_ids)) == 100
assert final_matrix.sum().min() >= 15

print("All checks passed.")

In [ ]:
print("Cases needed to achieve minimum coverage:",
      len(selected_ids))

In [ ]:
final_case_subjects = (
    df_normalized_diagnoses[
        df_normalized_diagnoses["hadm_id"].isin(selected_ids)
    ][["subject_id", "hadm_id"]]
    .drop_duplicates()
)

print("Final admissions:", final_case_subjects["hadm_id"].nunique())
print("Unique patients:", final_case_subjects["subject_id"].nunique())

In [ ]:
repeated_patients = (
    final_case_subjects
    .groupby("subject_id")["hadm_id"]
    .nunique()
    .reset_index(name="number_of_admissions")
)

repeated_patients = repeated_patients[
    repeated_patients["number_of_admissions"] > 1
]

print("Patients with multiple selected admissions:", len(repeated_patients))

if len(repeated_patients) > 0:
    print(repeated_patients.to_string(index=False))

In [ ]:
# Extracting the subject_id and hadm_id of the final 100 cases 
final_100_cases = (
    df_normalized_diagnoses[
        df_normalized_diagnoses["hadm_id"].isin(selected_ids)
    ][["subject_id", "hadm_id"]]
    .drop_duplicates()
    .sort_values(["subject_id", "hadm_id"])
    .reset_index(drop=True)
)

assert len(final_100_cases) == 100
assert final_100_cases["subject_id"].nunique() == 100
assert final_100_cases["hadm_id"].nunique() == 100

final_100_cases.head()

In [ ]:
# save the cases to a CSV
final_100_cases.to_csv(
    "final_100_evaluation_cases.csv",
    index=False
)

In [ ]:
# save the same frozen cohort to a BigQuery table: final_100_evaluation_cases
table_id = (
    "my-dissertation-project-502620."
    "Query_Results.final_100_evaluation_cases"
)

job = client.load_table_from_dataframe(
    final_100_cases,
    table_id
)

job.result()

print(f"Saved {len(final_100_cases)} cases to:")
print(table_id)

In [ ]:
# Only verification to confirm the tavle is correctly saved
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT subject_id) AS unique_patients,
    COUNT(DISTINCT hadm_id) AS unique_admissions
FROM `my-dissertation-project-502620.Query_Results.final_100_evaluation_cases`
"""

client.query(query).to_dataframe()

In [ ]:
def extract_demographics(subject_id):
    query = """
    SELECT
        subject_id,
        gender,
        anchor_age
    FROM `physionet-data.mimiciv_3_1_hosp.patients`
    WHERE subject_id = @subject_id
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id)
        ]
    )

    patient_demographics = client.query(query, job_config=job_config).to_dataframe()
    return patient_demographics
    
################Test
# patient_demographics= extract_demographics(00000000)
# print(patient_demographics)
################

def extract_admission(subject_id, hadm_id):
    query = """
    SELECT
        hadm_id, admittime, dischtime, admission_type, admission_location, discharge_location, insurance, language, marital_status, race, hospital_expire_flag
    FROM `physionet-data.mimiciv_3_1_hosp.admissions`
    WHERE subject_id = @subject_id
    AND hadm_id = @hadm_id
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_admission = client.query(query, job_config=job_config).to_dataframe()
    return patient_admission

# ################Test
# patient_admission= extract_admission (00000000, 00000000)
# print(patient_admission)
# ################

def extract_services (subject_id, hadm_id):
    query = """
    SELECT
        transfertime, prev_service, curr_service
    FROM `physionet-data.mimiciv_3_1_hosp.services`
    WHERE subject_id = @subject_id
    AND hadm_id = @hadm_id
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_services = client.query(query, job_config=job_config).to_dataframe()
    return patient_services

# ################Test
# patient_services= extract_services (00000000, 00000000)
# print(patient_services)
# ################
    
def extract_transfers (subject_id, hadm_id):
    query = """
    SELECT
        intime, outtime, eventtype, careunit
    FROM `physionet-data.mimiciv_3_1_hosp.transfers`
    WHERE subject_id = @subject_id
    AND hadm_id = @hadm_id
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_transfers = client.query(query, job_config=job_config).to_dataframe()
    return patient_transfers

# ################Test
# patient_transfers= extract_transfers (00000000, 00000000)
# print(patient_transfers)
# ################

def extract_procedures(subject_id, hadm_id):
    query = """
    SELECT
        p.seq_num,
        d.long_title
    FROM `physionet-data.mimiciv_3_1_hosp.procedures_icd` AS p
    JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_procedures` AS d
        ON p.icd_code = d.icd_code
        AND p.icd_version = d.icd_version
    WHERE p.subject_id = @subject_id
    AND p.hadm_id = @hadm_id
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_procedures = client.query(query, job_config=job_config).to_dataframe()
    return patient_procedures

# ################Test
# patient_procedures= extract_procedures (00000000, 00000000)
# print(patient_procedures)
# ################

    
def extract_medications(subject_id, hadm_id):
    query = """
    SELECT
        starttime, stoptime, drug, dose_val_rx, dose_unit_rx, route
    FROM `physionet-data.mimiciv_3_1_hosp.prescriptions`
    WHERE subject_id = @subject_id
    AND hadm_id = @hadm_id
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_medications = client.query(query, job_config=job_config).to_dataframe()
    return patient_medications

# ################Test
# patient_medications= extract_medications (00000000, 00000000)
# print(patient_medications)
# ################


def extract_laboratory_results(subject_id, hadm_id):
    query = """
    SELECT
        l.charttime, d.label, l.valuenum, l.valueuom, l.flag
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` AS l
    JOIN `physionet-data.mimiciv_3_1_hosp.d_labitems` AS d
        ON l.itemid = d.itemid
    WHERE l.subject_id = @subject_id
    AND l.hadm_id = @hadm_id
    AND l.valuenum IS NOT NULL
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_lab_results = client.query(query, job_config=job_config).to_dataframe()
    return patient_lab_results

# ################Test
# patient_lab_results= extract_laboratory_results (00000000, 00000000)
# print(patient_lab_results)
# ################

    
def extract_microbiology (subject_id, hadm_id): 
    query = """
    SELECT
        charttime, spec_type_desc, test_name, org_name, ab_name, interpretation
    FROM `physionet-data.mimiciv_3_1_hosp.microbiologyevents`
    WHERE subject_id = @subject_id
    AND hadm_id = @hadm_id
    AND org_name IS NOT NULL
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_microbiology = client.query(query, job_config=job_config).to_dataframe()
    return patient_microbiology

# ################Test
# patient_microbiology= extract_microbiology (00000000, 00000000)
# print(patient_microbiology)
# ################    

def extract_ed_data(subject_id, hadm_id):
    query = """
    SELECT
        e.intime, e.outtime, e.disposition, t.acuity, t.chiefcomplaint, v.charttime, v.temperature, v.heartrate, v.resprate, v.o2sat, v.sbp, v.dbp
    FROM `physionet-data.mimiciv_ed.edstays` AS e
    LEFT JOIN `physionet-data.mimiciv_ed.triage` AS t
        ON e.stay_id = t.stay_id
    LEFT JOIN `physionet-data.mimiciv_ed.vitalsign` AS v
        ON e.stay_id = v.stay_id
    WHERE e.subject_id = @subject_id
    AND e.hadm_id = @hadm_id
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_ed_data = client.query(query, job_config=job_config).to_dataframe()
    return patient_ed_data
    
# ################Test
# patient_ed_data= extract_ed_data (00000000, 00000000)
# print(patient_ed_data)
# ################    


def extract_icu_summary(subject_id, hadm_id):
    query = """
    SELECT 
        stay_id, first_careunit, last_careunit, intime, outtime, los
    FROM `physionet-data.mimiciv_3_1_icu.icustays`
    WHERE subject_id = @subject_id
    AND hadm_id = @hadm_id
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_icu_summary = client.query(query, job_config=job_config).to_dataframe()
    return patient_icu_summary

# ################Test
# patient_icu_summary= extract_icu_summary (00000000, 00000000)
# print(patient_icu_summary)
# ################   


In [ ]:
def extract_vitals(stay_id):
    query = """
    SELECT
        charttime,
        heart_rate,
        sbp,
        dbp,
        mbp,
        sbp_ni,
        dbp_ni,
        mbp_ni,
        resp_rate,
        temperature,
        temperature_site,
        spo2,
        glucose
    FROM `physionet-data.mimiciv_3_1_derived.vitalsign`
    WHERE stay_id = @stay_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("stay_id", "INT64", stay_id)
        ]
    )
    patient_vitals = client.query(query, job_config=job_config).to_dataframe()
    return patient_vitals


def extract_chemistry(hadm_id):
    query = """
    SELECT
        charttime,
        albumin,
        globulin,
        total_protein,
        aniongap,
        bicarbonate,
        bun,
        calcium,
        chloride,
        creatinine,
        glucose,
        sodium,
        potassium
    FROM `physionet-data.mimiciv_3_1_derived.chemistry`
    WHERE hadm_id = @hadm_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_chemistry = client.query(query, job_config=job_config).to_dataframe()
    return patient_chemistry

def extract_complete_blood_count(hadm_id):
    query = """
    SELECT
        charttime,
        hematocrit,
        hemoglobin,
        mch,
        mchc,
        mcv,
        platelet,
        rbc,
        rdw,
        rdwsd,
        wbc
    FROM `physionet-data.mimiciv_3_1_derived.complete_blood_count`
    WHERE hadm_id = @hadm_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_complete_blood_count = client.query(query, job_config=job_config).to_dataframe()
    return patient_complete_blood_count


def extract_blood_differential(hadm_id):
    query = """
    SELECT
        charttime,
        wbc,
        basophils_abs,
        eosinophils_abs,
        lymphocytes_abs,
        monocytes_abs,
        neutrophils_abs,
        basophils,
        eosinophils,
        lymphocytes,
        monocytes,
        neutrophils,
        atypical_lymphocytes,
        bands,
        immature_granulocytes,
        metamyelocytes,
        nrbc
    FROM `physionet-data.mimiciv_3_1_derived.blood_differential`
    WHERE hadm_id = @hadm_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_blood_differential = client.query(query, job_config=job_config).to_dataframe()
    return patient_blood_differential



def extract_coagulation(hadm_id):
    query = """
    SELECT
        charttime,
        d_dimer,
        fibrinogen,
        thrombin,
        inr,
        pt,
        ptt
    FROM `physionet-data.mimiciv_3_1_derived.coagulation`
    WHERE hadm_id = @hadm_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_coagulation = client.query(query, job_config=job_config).to_dataframe()
    return patient_coagulation



def extract_blood_gas(hadm_id):
    query = """
    SELECT
        charttime,
        specimen,
        so2,
        po2,
        pco2,
        fio2_chartevents,
        fio2,
        aado2,
        aado2_calc,
        pao2fio2ratio,
        ph,
        baseexcess,
        bicarbonate,
        totalco2,
        hematocrit,
        hemoglobin,
        carboxyhemoglobin,
        methemoglobin,
        chloride,
        calcium,
        temperature,
        potassium,
        sodium,
        lactate,
        glucose
    FROM `physionet-data.mimiciv_3_1_derived.bg`
    WHERE hadm_id = @hadm_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_blood_gas = client.query(query, job_config=job_config).to_dataframe()
    return patient_blood_gas


def extract_cardiac_markers(hadm_id):
    query = """
    SELECT
        charttime,
        troponin_t,
        ck_mb,
        ntprobnp
    FROM `physionet-data.mimiciv_3_1_derived.cardiac_marker`
    WHERE hadm_id = @hadm_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_cardiac_markers = client.query(query, job_config=job_config).to_dataframe()
    return patient_cardiac_markers


def extract_enzymes(hadm_id):
    query = """
    SELECT
        charttime,
        alt,
        alp,
        ast,
        amylase,
        bilirubin_total,
        bilirubin_direct,
        bilirubin_indirect,
        ck_cpk,
        ck_mb,
        ggt,
        ld_ldh
    FROM `physionet-data.mimiciv_3_1_derived.enzyme`
    WHERE hadm_id = @hadm_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)
        ]
    )
    patient_enzymes = client.query(query, job_config=job_config).to_dataframe()
    return patient_enzymes


def extract_urine_output(stay_id):
    query = """
    SELECT
        charttime,
        urineoutput
    FROM `physionet-data.mimiciv_3_1_derived.urine_output`
    WHERE stay_id = @stay_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("stay_id", "INT64", stay_id)
        ]
    )
    patient_urine_output = client.query(query, job_config=job_config).to_dataframe()
    return patient_urine_output

def extract_ventilation(stay_id):
    query = """
    SELECT
        starttime,
        endtime,
        ventilation_status
    FROM `physionet-data.mimiciv_3_1_derived.ventilation`
    WHERE stay_id = @stay_id
    ORDER BY starttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("stay_id", "INT64", stay_id)
        ]
    )
    patient_ventilation = client.query(query, job_config=job_config).to_dataframe()
    return patient_ventilation

def extract_oxygen_delivery(stay_id):
    query = """
    SELECT
        charttime,
        o2_flow,
        o2_flow_additional,
        o2_delivery_device_1,
        o2_delivery_device_2,
        o2_delivery_device_3,
        o2_delivery_device_4
    FROM `physionet-data.mimiciv_3_1_derived.oxygen_delivery`
    WHERE stay_id = @stay_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("stay_id", "INT64", stay_id)
        ]
    )
    patient_oxygen_delivery = client.query(query, job_config=job_config).to_dataframe()
    return patient_oxygen_delivery


#////////////////////

import re
import pandas as pd
from collections import Counter

def analyse_radiology_sections(radiology_df):
    """
    Analyse section headings appearing in raw radiology reports.

    Expects a DataFrame containing a 'text' column.
    Returns:
        1. A DataFrame with each detected heading and its frequency.
        2. A dictionary containing example reports for each heading.
    """

    section_counter = Counter()
    examples = {}

    # A heading must start at the beginning of a line and end with ":"
    heading_pattern = re.compile(
        r"^\s*([A-Z][A-Z0-9 /,_().'-]*):\s*",
        re.MULTILINE
    )

    for _, row in radiology_df.iterrows():

        text = row["text"]

        if pd.isna(text):
            continue

        # Find headings in this report
        headings = heading_pattern.findall(text)

        # Clean heading names
        headings = [
            re.sub(r"\s+", " ", h).strip()
            for h in headings
        ]

        # Count a section only once per report
        unique_headings = set(headings)

        for heading in unique_headings:
            section_counter[heading] += 1

            # Save one example report for later inspection
            if heading not in examples:
                examples[heading] = text

    section_summary = pd.DataFrame(
        section_counter.items(),
        columns=["section", "number_of_reports"]
    )

    section_summary = section_summary.sort_values(
        "number_of_reports",
        ascending=False
    ).reset_index(drop=True)

    return section_summary, examples




    
# Extracting radiology reports with redaction
def extract_radiology_reports(hadm_id):
    query = """
    SELECT charttime, text
    FROM `physionet-data.mimiciv_note.radiology`
    WHERE hadm_id = @hadm_id
    ORDER BY charttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id)]
    )
    patient_radiology_reports = client.query(query, job_config=job_config).to_dataframe()
    if not patient_radiology_reports.empty:
        patient_radiology_reports["text"] = patient_radiology_reports["text"].apply(redact_radiology_report)

        #////////////////////
        #### just to investigate
        section_summary, section_examples = analyse_radiology_sections(patient_radiology_reports)
        print(section_summary.to_string(index=False))
    return patient_radiology_reports




def extract_ecg(subject_id, admittime, dischtime):
    """
    Extract ECG recordings during a hospital admission.

    Parameters
    ----------
    subject_id : int
        Patient identifier.
    admittime : datetime
        Admission start time.
    dischtime : datetime
        Admission end time.

    Returns
    -------
    pandas.DataFrame
        ECG recordings with machine interpretation and measurements.
    """

    query = """
    SELECT
        rl.ecg_time,

        -- Machine-generated interpretation
        mm.report_0,
        mm.report_1,
        mm.report_2,
        mm.report_3,
        mm.report_4,
        mm.report_5,
        mm.report_6,
        mm.report_7,
        mm.report_8,
        mm.report_9,
        mm.report_10,
        mm.report_11,
        mm.report_12,
        mm.report_13,
        mm.report_14,
        mm.report_15,
        mm.report_16,
        mm.report_17,

        -- ECG measurements
        mm.rr_interval,
        mm.p_onset,
        mm.p_end,
        mm.qrs_onset,
        mm.qrs_end,
        mm.t_end,
        mm.p_axis,
        mm.qrs_axis,
        mm.t_axis

    FROM `physionet-data.mimiciv_ecg.record_list` AS rl

    INNER JOIN `physionet-data.mimiciv_ecg.machine_measurements` AS mm
        ON rl.study_id = mm.study_id

    WHERE
        rl.subject_id = @subject_id
        AND rl.ecg_time BETWEEN @admittime AND @dischtime

    ORDER BY rl.ecg_time
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("admittime", "DATETIME", admittime),
            bigquery.ScalarQueryParameter("dischtime", "DATETIME", dischtime),
        ]
    )

    patient_ecg = client.query(query, job_config=job_config).to_dataframe()

    return patient_ecg


def extract_medication_administration(subject_id, hadm_id):
    """
    Extract medication administration records (eMAR) for a hospital admission.
    This function will extract medications that were actually administered during the admission (what the patient actually received).
    It's more informative than prescriptions alone (what was ordered).
    Parameters
    ----------
    subject_id : int
        Patient identifier.

    hadm_id : int
        Hospital admission identifier.

    Returns
    -------
    pandas.DataFrame
        Medication administrations with administration details.
    """

    query = """
    SELECT
        e.charttime,
        e.medication,
        e.event_txt,
        e.scheduletime,
        e.storetime,

        d.dose_given,
        d.dose_given_unit,
        d.route,
        d.site,
        d.infusion_complete

    FROM `physionet-data.mimiciv_3_1_hosp.emar` AS e

    LEFT JOIN `physionet-data.mimiciv_3_1_hosp.emar_detail` AS d
        ON e.emar_id = d.emar_id

    WHERE
        e.subject_id = @subject_id
        AND e.hadm_id = @hadm_id

    ORDER BY e.charttime
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id),
        ]
    )

    patient_medication_administration = (client.query(query, job_config=job_config).to_dataframe() )

    return patient_medication_administration



def extract_inputevents(stay_id):
    """
    Extract ICU input events (IV fluids, medications, blood products,
    nutrition, etc.) for an ICU stay.

    Parameters
    ----------
    stay_id : int
        ICU stay identifier.

    Returns
    -------
    pandas.DataFrame
        ICU input events with descriptive item labels.
    """

    query = """
    SELECT
        ie.starttime,
        ie.endtime,
        d.label,
        ie.ordercategoryname,
        ie.secondaryordercategoryname,
        ie.ordercomponenttypedescription,
        ie.amount,
        ie.amountuom,
        ie.originalamount,
        ie.originalrate,
        ie.rate,
        ie.rateuom,
        ie.totalamount,
        ie.totalamountuom,
        ie.patientweight,
        ie.statusdescription

    FROM `physionet-data.mimiciv_3_1_icu.inputevents` AS ie

    INNER JOIN `physionet-data.mimiciv_3_1_icu.d_items` AS d
        ON ie.itemid = d.itemid

    WHERE ie.stay_id = @stay_id

    ORDER BY ie.starttime
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "stay_id",
                "INT64",
                stay_id
            )
        ]
    )

    patient_inputevents = client.query(query, job_config=job_config).to_dataframe()

    return patient_inputevents



def extract_procedureevents(stay_id):
    """
    Extract ICU procedure events performed during an ICU stay.

    Parameters
    ----------
    stay_id : int
        ICU stay identifier.

    Returns
    -------
    pandas.DataFrame
        ICU procedure events with descriptive procedure labels.
    """

    query = """
    SELECT
        pe.starttime,
        pe.endtime,
        d.label,
        pe.value,
        pe.valueuom,
        pe.location,
        pe.locationcategory,
        pe.ordercategoryname,
        pe.statusdescription

    FROM `physionet-data.mimiciv_3_1_icu.procedureevents` AS pe

    INNER JOIN `physionet-data.mimiciv_3_1_icu.d_items` AS d
        ON pe.itemid = d.itemid

    WHERE pe.stay_id = @stay_id

    ORDER BY pe.starttime
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "stay_id",
                "INT64",
                stay_id
            )
        ]
    )

    patient_procedureevents = (
        client.query(
            query,
            job_config=job_config
        ).to_dataframe()
    )

    return patient_procedureevents


def extract_outputevents(stay_id):
    """
    Extract ICU output events (urine, drains, chest tubes, dialysis,
    gastrointestinal output, etc.) for an ICU stay.

    Parameters
    ----------
    stay_id : int
        ICU stay identifier.

    Returns
    -------
    pandas.DataFrame
        ICU output events with descriptive output labels.
    """

    query = """
    SELECT
        oe.charttime,
        d.label,
        oe.value,
        oe.valueuom,
        oe.storetime

    FROM `physionet-data.mimiciv_3_1_icu.outputevents` AS oe

    INNER JOIN `physionet-data.mimiciv_3_1_icu.d_items` AS d
        ON oe.itemid = d.itemid

    WHERE oe.stay_id = @stay_id

    ORDER BY oe.charttime
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "stay_id",
                "INT64",
                stay_id
            )
        ]
    )

    patient_outputevents = (
        client.query(
            query,
            job_config=job_config
        ).to_dataframe()
    )

    return patient_outputevents    




## in the main 
# patient_icu_summary = extract_icu_summary (00000000, 00000000) 
# if not patient_icu_summary.empty: 
#     stay_id = int(patient_icu_summary["stay_id"].iloc[0])
#     patient_vitals = extract_vitals(stay_id) 
#     print (patient_vitals)
#     print("------------------")
#     patient_urine_output = extract_urine_output(stay_id)
#     print (patient_urine_output)
#     print("------------------")
#     patient_ventilation = extract_ventilation(stay_id)
#     print (patient_ventilation)
#     print("------------------")
#     patient_oxygen_delivery = extract_oxygen_delivery(stay_id)
#     print (patient_oxygen_delivery)
#     print("------------------")
#     patient_inputevents = extract_inputevents(stay_id)
#     print (patient_inputevents)
#     print("------------------")
#     patient_outputevents = extract_outputevents(stay_id)
#     print (patient_outputevents)
#     print("------------------")
#     patient_procedureevents = extract_procedureevents(stay_id)
#     print (patient_procedureevents)
#     print("------------------")

    
# ################Test  (00000000, 00000000)
# print("**********************")
# patient_chemistry= extract_chemistry (00000000) 
# print(patient_chemistry)
# print("**********************")
# patient_complete_blood_count = extract_complete_blood_count(00000000)
# print(patient_complete_blood_count)
# print("**********************")
# patient_blood_differential=extract_blood_differential(00000000)
# print(patient_blood_differential)
# print("**********************")
# patient_coagulation = extract_coagulation(00000000)
# print(patient_coagulation)
# print("**********************")
# patient_blood_gas = extract_blood_gas(00000000)
# print(patient_blood_gas)
# print("**********************")
# patient_cardiac_markers = extract_cardiac_markers(00000000)
# print(patient_cardiac_markers)
# print("**********************")
# patient_enzymes = extract_enzymes(00000000)
# print(patient_enzymes)
# print("**********************")
# patient_radiology_reports = extract_radiology_reports(00000000)
# print(patient_radiology_reports)
# print("**********************")
# patient_admission=extract_admission(00000000, 00000000)
# row = patient_admission.iloc[0]
# patient_ecg= extract_ecg(00000000, row["admittime"], row["dischtime"])
# print(patient_ecg)
# print("**********************")
# patient_medication_administration = extract_medication_administration(00000000, 00000000)
# # another pair of subject_id,hadm_id that returns values: (00000000, 00000000)
# print(patient_medication_administration)
# print("**********************")

In [ ]:
# Extracting the target variables: the diagnoses and the discharge summary
def extract_diagnoses(subject_id, hadm_id):
    """
    Extract the final ICD diagnoses for a hospital admission.

    Parameters
    ----------
    subject_id : int
        Patient identifier.
    hadm_id : int
        Hospital admission identifier.

    Returns
    -------
    pandas.DataFrame
        Final diagnoses assigned during the admission.
    """

    query = """
    SELECT
        d.seq_num,
        d.icd_code,
        d.icd_version,
        i.long_title
    FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` AS d

    LEFT JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses` AS i
        ON d.icd_code = i.icd_code
        AND d.icd_version = i.icd_version

    WHERE
        d.subject_id = @subject_id
        AND d.hadm_id = @hadm_id

    ORDER BY d.seq_num
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "subject_id",
                "INT64",
                subject_id
            ),
            bigquery.ScalarQueryParameter(
                "hadm_id",
                "INT64",
                hadm_id
            ),
        ]
    )

    patient_diagnoses = (
        client.query(query, job_config=job_config)
        .to_dataframe()
    )

    return patient_diagnoses


def extract_discharge_summary(subject_id, hadm_id):
    """
    Extract the discharge summary for a hospital admission.

    Parameters
    ----------
    subject_id : int
        Patient identifier.
    hadm_id : int
        Hospital admission identifier.

    Returns
    -------
    pandas.DataFrame
        Discharge summary note.
    """

    query = """
    SELECT
        note_id,
        charttime,
        storetime,
        text
    FROM `physionet-data.mimiciv_note.discharge`

    WHERE
        subject_id = @subject_id
        AND hadm_id = @hadm_id

    ORDER BY charttime
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id),
        ]
    )

    patient_discharge_summary = (
        client.query(query, job_config=job_config).to_dataframe()
    )
    if not patient_discharge_summary.empty:
        patient_discharge_summary["text"] = patient_discharge_summary["text"].str.replace(r"\n", "\n", regex=False)
    return patient_discharge_summary

In [ ]:
import pandas as pd
import numpy as np

def summarise_numeric(df, time_col):
    """
    Summarise a time-series dataframe containing numeric measurements.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe containing one or more numeric variables.
    time_col : str
        Name of the timestamp column.

    Returns
    -------
    pandas.DataFrame
        One row per numeric variable with summary statistics.
    """

    if df.empty:
        return pd.DataFrame()

    df = df.sort_values(time_col)

    numeric_cols = [
        col for col in df.columns
        if col != time_col and pd.api.types.is_numeric_dtype(df[col])
    ]

    summary = []

    for col in numeric_cols:

        values = df[[time_col, col]].dropna()

        if values.empty:
            continue

        summary.append({
            "Measurement": col,
            "N": len(values),
            "First": float(values.iloc[0][col]),
            "Latest": float(values.iloc[-1][col]),
            "Minimum": float(values[col].min()),
            "Maximum": float(values[col].max()),
            "Mean": round(float(values[col].mean()), 2),
            "Median": round(float(values[col].median()), 2),
        })

    return pd.DataFrame(summary)



def summarise_medications(patient_medications):
    """
    Summarise prescribed medications for a hospital admission.

    Parameters
    ----------
    patient_medications : pandas.DataFrame
        Output of extract_medications().

    Returns
    -------
    pandas.DataFrame
        One row per unique medication, dose and route.
    """

    if patient_medications.empty:
        return pd.DataFrame()

    summary = (
        patient_medications
        .groupby(
            [
                "drug",
                "dose_val_rx",
                "dose_unit_rx",
                "route"
            ],
            dropna=False
        )
        .agg(
            first_prescribed=("starttime", "min"),
            last_prescribed=("stoptime", "max"),
            number_of_prescriptions=("drug", "count")
        )
        .reset_index()
        .sort_values(
            ["drug", "first_prescribed"]
        )
    )

    return summary



def summarise_medication_administration(patient_medication_administration):
    """
    Summarise medication administrations for a hospital admission.

    Parameters
    ----------
    patient_medication_administration : pandas.DataFrame
        Output of extract_medication_administration().

    Returns
    -------
    pandas.DataFrame
        One row per unique medication, dose and route.
    """

    if patient_medication_administration.empty:
        return pd.DataFrame()

    summary = (
        patient_medication_administration
        .groupby(
            [
                "medication",
                "dose_given",
                "dose_given_unit",
                "route"
            ],
            dropna=False
        )
        .agg(
            first_administration=("charttime", "min"),
            last_administration=("charttime", "max"),
            number_of_administrations=("medication", "count")
        )
        .reset_index()
        .sort_values(
            ["medication", "first_administration"]
        )
    )

    return summary



def summarise_microbiology(patient_microbiology):
    """
    Summarise microbiology results for a hospital admission.

    Parameters
    ----------
    patient_microbiology : pandas.DataFrame
        Output of extract_microbiology().

    Returns
    -------
    pandas.DataFrame
        One row per specimen/organism combination with antibiotic
        susceptibility profile.
    """

    if patient_microbiology.empty:
        return pd.DataFrame()

    # Build "Antibiotic (Interpretation)" strings
    patient_microbiology = patient_microbiology.copy()

    patient_microbiology["susceptibility"] = (
        patient_microbiology["ab_name"].fillna("")
        + " ("
        + patient_microbiology["interpretation"].fillna("")
        + ")"
    )

    summary = (
        patient_microbiology
        .groupby(
            [
                "charttime",
                "spec_type_desc",
                "org_name"
            ],
            dropna=False
        )
        .agg(
            susceptibility_profile=(
                "susceptibility",
                lambda x: ", ".join(
                    sorted(set(filter(None, x)))
                )
            )
        )
        .reset_index()
        .sort_values("charttime")
    )

    return summary



def summarise_ecg(patient_ecg):
    """
    Summarise ECG recordings by combining machine interpretation
    fields into a single interpretation string.

    Parameters
    ----------
    patient_ecg : pandas.DataFrame
        Output of extract_ecg().

    Returns
    -------
    pandas.DataFrame
        One row per ECG recording.
    """

    if patient_ecg.empty:
        return pd.DataFrame()

    patient_ecg = patient_ecg.copy()

    report_cols = [f"report_{i}" for i in range(18)]

    patient_ecg["interpretation"] = patient_ecg[report_cols].apply(
        lambda row: "; ".join(
            str(x).strip()
            for x in row
            if pd.notna(x) and str(x).strip() != ""
        ),
        axis=1
    )

    summary = patient_ecg[
        [
            "ecg_time",
            "interpretation",
            "rr_interval",
            "p_axis",
            "qrs_axis",
            "t_axis",
        ]
    ].copy()

    return summary


In [ ]:
import re

def regex_extracter(text, regex):
    match = regex.search(text)
    if match:
        return match.group(0), True
    return text, False

def extract_history(text):
    """HPI only -- safe for case input."""
    text = text.replace("\n", " ")
    pe_strings = ["physical exam:", "physical examination:", "physical ___:",
                  "pe:", "pe ___:", "(?:pertinent|___) results:", "hospital course:"]
    success = False
    i = 0
    while not success and i < len(pe_strings):
        regex = re.compile(f"(?:history|___) of present(?:ing)? illness:.*?{pe_strings[i]}",
                            re.IGNORECASE | re.DOTALL)
        text, success = regex_extracter(text, regex)
        i += 1
    if not success:
        return ""
    text = re.sub(re.compile("history of present(?:ing)? illness:", re.IGNORECASE), "", text)
    for pe_str in pe_strings:
        text = re.sub(re.compile(pe_str, re.IGNORECASE), "", text)
    return text.strip()

def extract_physical_examination(text):
    """Physical exam findings only -- safe for case input."""
    text = text.replace("\n", " ")
    pe_strings = ["physical exam:", "physical examination:", "physical ___:", "pe:", "pe ___:", "pertinent results:"]
    success = False
    i = 0
    while not success and i < len(pe_strings):
        terminal_str = "pertinent results:" if "pertinent results:" in text.lower() else "brief hospital course:"
        regex = re.compile(f"{pe_strings[i]}.*?{terminal_str}", re.IGNORECASE | re.DOTALL)
        text, success = regex_extracter(text, regex)
        i += 1
    if not success:
        return ""
    for pe_str in pe_strings:
        text = re.sub(re.compile(pe_str, re.IGNORECASE), "", text)
    text = re.sub(re.compile("pertinent results:", re.IGNORECASE), "", text)
    text = re.sub(re.compile("brief hospital course:", re.IGNORECASE), "", text)
    text = re.sub(re.compile("(?:at|upon|on) discharge.*", re.IGNORECASE), "", text)
    return text.strip()

REDACTED_RAD_SECTIONS = ["IMPRESSION", "CONCLUSION", "COMMENT", "CLINICAL HISTORY",
                          "MEDICAL HISTORY", "CLINICAL INFORMATION", "CLINICAL INDICATION",
                          "INDICATION", "REASON", "COMPARISON", "COMPARISONS", "HISTORY", "REFERENCE", "DATE"]

def parse_report(report):
    lines = report.strip().split("\n")
    if lines and lines[0].isupper() and lines[0].strip()[-1] != ":":
        lines[0] = lines[0].strip() + ":"
    for i, line in enumerate(lines):
        if line.isupper() and ":" not in line:
            lines[i] = line.strip() + ":"
    report = "\n".join(lines)
    pattern = r"(?m)^([A-Z \t,._-]+):((?:(?!^[A-Z \t,._-]+:).)*)"
    sections = re.findall(pattern, report, re.DOTALL)
    return {sec[0].strip(): sec[1].strip() for sec in sections}

def redact_radiology_report(text):
    """Strips IMPRESSION/CONCLUSION, keeps FINDINGS and technical detail."""
    sections = parse_report(text)
    kept = ""
    for field, content in sections.items():
        if any(field.startswith(bad) for bad in REDACTED_RAD_SECTIONS):
            continue
        if content:
            kept += f"{field}:\n{content}\n\n"
    return kept.strip()

In [ ]:
def add_section(context_parts, title, data):

    """
    Add a formatted section to the patient context.

    Parameters
    ----------
    context_parts : list
        List that accumulates all sections.

    title : str
        Section title.

    dataframe : pandas.DataFrame
        Data to include in the section.
    """

    if data is None:
        return

    if isinstance(data, pd.DataFrame):
        if data.empty:
            return
        content = data.to_string(index=False)

    elif isinstance(data, dict):
        if len(data) == 0:
            return
        content = str(data)

    else:
        content = str(data)

    context_parts.append("=" * 80)
    context_parts.append(title.upper())
    context_parts.append("=" * 80)
    context_parts.append(content)
    context_parts.append("")



def add_section1(context_parts, title, data):
    if data is None:
        return

    if isinstance(data, pd.DataFrame):
        if data.empty:
            return
        if "text" in data.columns:
            # Free-text note data (discharge summaries, radiology reports):
            # to_string() escapes embedded real newlines as literal "\n" to
            # preserve tabular alignment -- bypass it and format manually.
            rows = []
            for _, row in data.iterrows():
                meta = row.drop("text").to_dict()
                rows.append(f"{meta}\n{row['text']}")
            content = "\n\n".join(rows)
        else:
            content = data.to_string(index=False)

    elif isinstance(data, dict):
        if len(data) == 0:
            return
        content = str(data)

    else:
        content = str(data)

    context_parts.append("=" * 80)
    context_parts.append(title.upper())
    context_parts.append("=" * 80)
    context_parts.append(content)
    context_parts.append("")






def build_patient_context(subject_id, hadm_id):

    #### The Etraction phase: Calling the extraction/summarisation functions
    
    patient_demographics = extract_demographics(subject_id)
    
    patient_admission = extract_admission(subject_id, hadm_id)
    
    patient_services = extract_services (subject_id, hadm_id)
    
    patient_transfers = extract_transfers (subject_id, hadm_id)
    
    patient_procedures = extract_procedures(subject_id, hadm_id)
    
    patient_medications = extract_medications(subject_id, hadm_id)
    summarised_patient_medications = summarise_medications(patient_medications)

    patient_medication_administration = extract_medication_administration (subject_id, hadm_id)
    summarised_patient_medication_administration = summarise_medication_administration(patient_medication_administration)
    
    patient_laboratory_results = extract_laboratory_results(subject_id, hadm_id)

    patient_microbiology = extract_microbiology (subject_id, hadm_id)
    summarised_patient_microbiology = summarise_microbiology (patient_microbiology)
    
    patient_ed_data = extract_ed_data(subject_id, hadm_id)
    
    patient_icu_summary = extract_icu_summary(subject_id, hadm_id)

    patient_chemistry = extract_chemistry (hadm_id)
    summarised_patient_chemistry = summarise_numeric (patient_chemistry, time_col="charttime")
    
    patient_complete_blood_count = extract_complete_blood_count (hadm_id)
    summarised_patient_complete_blood_count = summarise_numeric (patient_complete_blood_count, time_col="charttime")
    
    patient_blood_differential = extract_blood_differential (hadm_id)
    summarised_patient_blood_differential = summarise_numeric (patient_blood_differential, time_col="charttime")
    
    patient_coagulation = extract_coagulation (hadm_id)
    summarised_patient_coagulation = summarise_numeric (patient_coagulation, time_col="charttime")
    
    patient_blood_gas = extract_blood_gas (hadm_id)
    summarised_patient_blood_gas = summarise_numeric (patient_blood_gas, time_col="charttime")
    
    patient_cardiac_markers = extract_cardiac_markers (hadm_id)
    summarised_patient_cardiac_markers = summarise_numeric (patient_cardiac_markers, time_col="charttime")
    
    patient_enzymes = extract_enzymes (hadm_id)
    summarised_patient_enzymes = summarise_numeric (patient_enzymes, time_col="charttime")

    patient_diagnoses = extract_diagnoses(subject_id, hadm_id)

    patient_discharge_summary = extract_discharge_summary(subject_id, hadm_id)

    # Redacted HPI + physical exam, safe for case_input (discharge summary itself stays in target_parts only)
    if not patient_discharge_summary.empty:
        full_text = " ".join(patient_discharge_summary["text"].astype(str))
        patient_hpi = extract_history(full_text)
        patient_physical_exam = extract_physical_examination(full_text)
    else:
        patient_hpi = ""
        patient_physical_exam = ""

    
    # If the patient has no ICU stay, variables such as patient_vitals,..etc won't exist, so we need to initialise them
    patient_vitals = pd.DataFrame()
    summarised_patient_vitals = pd.DataFrame()
    patient_urine_output = pd.DataFrame()
    patient_ventilation = pd.DataFrame()
    patient_oxygen_delivery = pd.DataFrame()
    patient_inputevents = pd.DataFrame()
    patient_procedureevents = pd.DataFrame()
    patient_outputevents = pd.DataFrame()
    
    if not patient_icu_summary.empty: 
        stay_id = int(patient_icu_summary["stay_id"].iloc[0])

        patient_vitals = extract_vitals(stay_id)
        summarised_patient_vitals = summarise_numeric(patient_vitals, time_col="charttime")

        patient_urine_output = extract_urine_output (stay_id)
    
        patient_ventilation = extract_ventilation (stay_id)
    
        patient_oxygen_delivery = extract_oxygen_delivery (stay_id)
    
        patient_inputevents = extract_inputevents (stay_id)
    
        patient_procedureevents = extract_procedureevents (stay_id)

        patient_outputevents = extract_outputevents (stay_id)
        
    patient_radiology_reports = extract_radiology_reports (hadm_id)
    
    if not patient_admission.empty:
        row = patient_admission.iloc[0]
        patient_ecg = extract_ecg (subject_id, row["admittime"], row["dischtime"])
        summarised_patient_ecg = summarise_ecg (patient_ecg)
    else:
        patient_ecg = pd.DataFrame()
        summarised_patient_ecg = pd.DataFrame()
    
    


    #### The Building phase: 
    context_parts = []

    add_section(context_parts, "Patient Demographics", patient_demographics)
    # Adding text block of HPI and PE 
    add_section(context_parts, "History of Present Illness", patient_hpi)
    add_section(context_parts, "Physical Examination", patient_physical_exam)
    
    add_section(context_parts, "Admission", patient_admission)
    add_section(context_parts, "Services", patient_services)
    add_section(context_parts, "Transfers", patient_transfers)
    add_section(context_parts, "Procedures", patient_procedures)
    add_section(context_parts, "Medications", summarised_patient_medications)
    add_section(context_parts, "Medication Administration", summarised_patient_medication_administration)
    add_section(context_parts, "Laboratory Results", patient_laboratory_results)
    add_section(context_parts, "Microbiology", summarised_patient_microbiology)
    add_section(context_parts, "Emergency Department Data", patient_ed_data)
    add_section(context_parts, "ICU Summary", patient_icu_summary)
    add_section(context_parts, "Patient Chemistry", summarised_patient_chemistry)
    add_section(context_parts, "Complete Blood Count", summarised_patient_complete_blood_count)
    add_section(context_parts, "Blood Differential", summarised_patient_blood_differential)
    add_section(context_parts, "Coagulation", summarised_patient_coagulation)
    add_section(context_parts, "Blood Gas", summarised_patient_blood_gas)
    add_section(context_parts, "Cardiac Markers", summarised_patient_cardiac_markers)
    add_section(context_parts, "Enzymes", summarised_patient_enzymes)
    add_section(context_parts, "Patient Vitals", summarised_patient_vitals)
    add_section(context_parts, "Urine Output", patient_urine_output)
    add_section(context_parts, "Ventilation", patient_ventilation)
    add_section(context_parts, "Oxygen Delivery", patient_oxygen_delivery)
    add_section(context_parts, "Input Events", patient_inputevents)
    add_section(context_parts, "Procedure Events", patient_procedureevents)
    add_section(context_parts, "Output Events", patient_outputevents)
    add_section1(context_parts, "Radiology Reports", patient_radiology_reports)
    add_section(context_parts, "Patient ECG", summarised_patient_ecg)

    #### Building the ground truth variables: 
    target_parts = []
    add_section(target_parts, "Patient Diagnoses", patient_diagnoses)
    add_section1(target_parts, "Patient Discharge Summary", patient_discharge_summary)
    
    patient_context = "\n".join(context_parts)
    patient_target = "\n".join(target_parts)
    return patient_context, patient_target
    

    
import os

def save_patient_context(patient_context, subject_id, hadm_id, output_dir="patient_contexts"):
    """
    Save a patient context to a text file.

    Parameters
    ----------
    patient_context : str
        Output from build_patient_context().
    subject_id : int
        Patient identifier.
    hadm_id : int
        Hospital admission identifier.
    output_dir : str, optional
        Directory where patient contexts will be saved.
    """

    # Create the output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)

    # Create the filename
    filename = f"patient_{subject_id}_{hadm_id}.txt"
    filepath = os.path.join(output_dir, filename)

    # Save the file
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(patient_context)

    return filepath

    
def save_patient_target(patient_target, subject_id, hadm_id, output_dir="patient_contexts"):
    """
    Save a patient target (diagnoses and discharge summary) to a text file.

    Parameters
    ----------
    patient_target : str
        Output from build_patient_context().
    subject_id : int
        Patient identifier.
    hadm_id : int
        Hospital admission identifier.
    output_dir : str, optional
        Directory where patient contexts will be saved.
    """

    # Create the output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)

    # Create the filename
    filename = f"patient_{subject_id}_{hadm_id}_target.txt"
    filepath = os.path.join(output_dir, filename)

    # Save the file
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(patient_target)

    return filepath    
    
    

In [ ]:
patient_context, patient_target = build_patient_context(00000000, 00000000)
#patient_context
filepath1 = save_patient_context (patient_context,00000000, 00000000,"patient_contexts")

#patient_target
filepath2 = save_patient_target (patient_target,00000000, 00000000,"patient_contexts")
print(filepath1)
print(filepath2)

In [ ]:
# Retrieve the first 50  narrowed candidate cases that satisfy 4 specialties and select the unique patient id
query = """
SELECT
    subject_id,
    hadm_id
FROM (
    SELECT
        subject_id,
        hadm_id,
        num_specialties,
        ROW_NUMBER() OVER (
            PARTITION BY subject_id
            ORDER BY hadm_id
        ) AS rn
    FROM `Query_Results.narrowed_candidate_cases`
    WHERE num_specialties = 4
)
WHERE rn = 1
ORDER BY subject_id
LIMIT 50
"""

narrowed_candidate_cases = client.query(query).to_dataframe()

# Generate patient contexts and targets
for _, row in narrowed_candidate_cases.iterrows():

    subject_id = int(row["subject_id"])
    hadm_id = int(row["hadm_id"])

    print(f"Processing Subject {subject_id}, Admission {hadm_id}...")

    try:
        patient_context, patient_target = build_patient_context(
            subject_id,
            hadm_id
        )

        context_path = save_patient_context(
            patient_context,
            subject_id,
            hadm_id,
            "patient_contexts"
        )

        target_path = save_patient_target(
            patient_target,
            subject_id,
            hadm_id,
            "patient_targets"
        )

    except Exception as e:
        print(
            f"Failed for Subject {subject_id}, "
            f"Admission {hadm_id}: {e}"
        )

print("Finished generating 50 patient cases.")